# Data Cleaning — World Climbing Results

Cleans the scraped data (`athlete_profiles.csv`, `climbing_results.csv`) and
separates it for per-discipline analysis.

**Separation strategy**
- **Para-climbing** is pulled out entirely (identified by sport-class codes in `category_name`: AL/AU/RP/B#, 'visual impairment', or 'para' in the event name).
- **Discipline**: boulder, lead, speed, combined, boulder&lead each get their own files.
- **Age group**: senior (open), youth, and junior each get their own files.

**Outputs** → `../resources/clean/`
- `results_clean.csv` — full cleaned table with derived columns
- `athlete_profiles_clean.csv` — cleaned athlete table
- `para_all.csv` — all para-climbing results
- `{discipline}_{age_group}.csv` — one file per able-bodied discipline × age group


In [ ]:
import os
import re

import numpy as np
import pandas as pd


In [ ]:
profiles = pd.read_csv("../resources/athlete_profiles.csv")
results  = pd.read_csv("../resources/climbing_results.csv")

print("profiles:", profiles.shape)
print("results: ", results.shape)
results.head()

## 1. Clean Athlete Profiles

Note the heavy missingness in physical attributes — `height` and `arm_span`
are absent for the large majority of athletes, so they are kept for reference
but are not reliable model features.

In [ ]:
profiles = profiles.drop_duplicates(subset="athlete_id")
profiles["lastname"] = profiles["lastname"].fillna("")
profiles["birthday"] = pd.to_datetime(profiles["birthday"], errors="coerce")
profiles["age"] = pd.to_numeric(profiles["age"], errors="coerce")

pct_missing = (profiles.isnull().mean() * 100).round(1)
print("Missing % per column:")
print(pct_missing)

## 2. Flag Para-Climbing

In [ ]:
# Para sport-class codes appear in category_name (e.g. 'Men AL2', 'Women B2',
# 'Men RP1', 'Men visual impairment B2'). Able-bodied categories never match.
PARA_CAT = re.compile(r"\b(AL|AU|RP)-?\d|\bB[1-3]\b|visual impairment", re.IGNORECASE)

def is_para(row):
    cat = str(row["category_name"])
    event = str(row["event_name"]).lower()
    return bool(PARA_CAT.search(cat)) or ("para" in event)

results["is_para"] = results.apply(is_para, axis=1)
print(results["is_para"].value_counts())
print()
print("Sample para categories:")
print(results[results["is_para"]]["category_name"].value_counts().head(10))

## 3. Derive Age Group & Gender from Category

In [ ]:
def age_group(category):
    c = str(category).lower()
    if "youth" in c or "u15" in c or "u17" in c:
        return "youth"
    if "junior" in c or "u19" in c or "u20" in c or "u21" in c:
        return "junior"
    return "senior"  # open "Men" / "Women"

def gender_from_category(category):
    c = str(category).lower()
    if "women" in c or "female" in c:   # check first — 'women' contains 'men'
        return "female"
    if "men" in c or "male" in c:
        return "male"
    return None

results["age_group"] = results["category_name"].apply(age_group)
results["gender"] = results["category_name"].apply(gender_from_category)

print(results["age_group"].value_counts())
print()
print(results["gender"].value_counts(dropna=False))

## 4. Clean Rank, Date & Season

`rank == 999` is a sentinel for unranked/DNS/DNF and is converted to `NaN`.

In [ ]:
results["rank"] = pd.to_numeric(results["rank"], errors="coerce")
results.loc[results["rank"] >= 999, "rank"] = np.nan

results["date"] = pd.to_datetime(results["date"], errors="coerce")
results["season"] = pd.to_numeric(results["season"], errors="coerce").astype("Int64")

print("rank range:", results["rank"].min(), "to", results["rank"].max())
print("unranked (NaN) rows:", results["rank"].isnull().sum())
print("date range:", results["date"].min(), "to", results["date"].max())

## 5. Merge Athlete Attributes

Adds `country` (the key feature in the original model) plus physical attributes,
and computes age at the time of each competition.

In [ ]:
results = results.merge(
    profiles[["athlete_id", "country", "height", "arm_span", "birthday"]],
    on="athlete_id",
    how="left",
)

# Age at competition (more meaningful than current age)
results["age_at_comp"] = (
    (results["date"] - results["birthday"]).dt.days / 365.25
).round(1)

# Guard against bad source birthdays (negative / implausibly young ages)
results.loc[results["age_at_comp"] < 5, "age_at_comp"] = np.nan

# Drop internal id and the now-redundant birthday
results = results.drop(columns=["d_cat", "birthday"])
results.head()

## 6. Split & Save

In [ ]:
CLEAN_DIR = "../resources/clean"
os.makedirs(CLEAN_DIR, exist_ok=True)

# Full cleaned table + cleaned profiles
results.to_csv(f"{CLEAN_DIR}/results_clean.csv", index=False)
profiles.to_csv(f"{CLEAN_DIR}/athlete_profiles_clean.csv", index=False)

# Para-climbing pulled out entirely
para = results[results["is_para"]]
para.to_csv(f"{CLEAN_DIR}/para_all.csv", index=False)
print(f"para_all.csv: {len(para):,} rows")

# Able-bodied: one file per discipline x age group
able = results[~results["is_para"]]
for (disc, age), grp in able.groupby(["discipline", "age_group"]):
    safe_disc = disc.replace("&", "_and_")
    fname = f"{CLEAN_DIR}/{safe_disc}_{age}.csv"
    grp.to_csv(fname, index=False)
    print(f"{os.path.basename(fname)}: {len(grp):,} rows")